# Lesson 1: Your first agent with Amazon Bedrock

In [1]:
# Before you start, please run the following code to set up your environment.
# This code will reset the environment (if needed) and prepare the resources for the lesson.
# It does this by quickly running through all the code from the previous lessons.

!sh ./ro_shared_data/reset.sh

import os

#roleArn = os.environ['BEDROCKAGENTROLE']
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


Resetting environment (if nessesary)
Agent reset process completed.
Lambda reset process completed.
Guardrail reset process completed.
Environment reset complete.


## Start of the lesson

In [2]:
import boto3

In [3]:
bedrock_agent = boto3.client(service_name='bedrock-agent', region_name='us-west-2')

In [4]:
create_agent_response = bedrock_agent.create_agent(
    agentName='mugs-customer-support-agent',
    foundationModel='anthropic.claude-3-haiku-20240307-v1:0',
    instruction="""You are an advanced AI agent acting as a front line customer support agent.""",
    agentResourceRoleArn='arn:aws:iam::092413168457:role/Bedrock-agent-role'
)

In [5]:
create_agent_response

{'ResponseMetadata': {'RequestId': '9bae58ac-1281-4043-8e1d-5ab126d92e79',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Sun, 10 Aug 2025 20:56:32 GMT',
   'content-type': 'application/json',
   'content-length': '578',
   'connection': 'keep-alive',
   'x-amzn-requestid': '9bae58ac-1281-4043-8e1d-5ab126d92e79',
   'x-amz-apigw-id': 'PG4cJFIkvHcEq8g=',
   'x-amzn-trace-id': 'Root=1-68990780-5334bc5f6d93849816816130'},
  'RetryAttempts': 0},
 'agent': {'agentArn': 'arn:aws:bedrock:us-west-2:092413168457:agent/8SOPTO6T5R',
  'agentCollaboration': 'DISABLED',
  'agentId': '8SOPTO6T5R',
  'agentName': 'mugs-customer-support-agent',
  'agentResourceRoleArn': 'arn:aws:iam::092413168457:role/Bedrock-agent-role',
  'agentStatus': 'CREATING',
  'createdAt': datetime.datetime(2025, 8, 10, 20, 56, 32, 628770, tzinfo=tzlocal()),
  'foundationModel': 'anthropic.claude-3-haiku-20240307-v1:0',
  'idleSessionTTLInSeconds': 600,
  'instruction': 'You are an advanced AI agent acting as a front lin

In [6]:
agentId = create_agent_response['agent']['agentId']

In [7]:
from helper import *

In [8]:
wait_for_agent_status(
    agentId=agentId, 
    targetStatus='NOT_PREPARED'
)

Waiting for agent status of 'NOT_PREPARED'...
Agent status: NOT_PREPARED
Agent reached 'NOT_PREPARED' status.


In [9]:
bedrock_agent.prepare_agent(
    agentId=agentId
)

{'ResponseMetadata': {'RequestId': '2e23d733-7989-419f-9511-aa7d5a4e32c6',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Sun, 10 Aug 2025 20:56:42 GMT',
   'content-type': 'application/json',
   'content-length': '119',
   'connection': 'keep-alive',
   'x-amzn-requestid': '2e23d733-7989-419f-9511-aa7d5a4e32c6',
   'x-amz-apigw-id': 'PG4dsGcoPHcEaTQ=',
   'x-amzn-trace-id': 'Root=1-6899078a-7d2d7e3b0795cb98405c49ab'},
  'RetryAttempts': 0},
 'agentId': '8SOPTO6T5R',
 'agentStatus': 'PREPARING',
 'agentVersion': 'DRAFT',
 'preparedAt': datetime.datetime(2025, 8, 10, 20, 56, 42, 719643, tzinfo=tzlocal())}

In [10]:
wait_for_agent_status(
    agentId=agentId, 
    targetStatus='PREPARED'
)

Waiting for agent status of 'PREPARED'...
Agent status: PREPARED
Agent reached 'PREPARED' status.


In [11]:
create_agent_alias_response = bedrock_agent.create_agent_alias(
    agentId=agentId,
    agentAliasName='MyAgentAlias',
)

agentAliasId = create_agent_alias_response['agentAlias']['agentAliasId']

wait_for_agent_alias_status(
    agentId=agentId,
    agentAliasId=agentAliasId,
    targetStatus='PREPARED'
)

Waiting for agent alias status of 'PREPARED'...
Agent alias status: CREATING
Agent alias status: CREATING
Agent alias status: PREPARED
Agent alias reached status 'PREPARED'


In [12]:
bedrock_agent_runtime = boto3.client(service_name='bedrock-agent-runtime', region_name='us-west-2')

In [13]:
import uuid

In [14]:
message = "Hello, I bought a mug from your store yesterday, and it broke. I want to return it."

sessionId = str(uuid.uuid4())

invoke_agent_response = bedrock_agent_runtime.invoke_agent(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    endSession=False,
    enableTrace=True,
)

In [15]:
event_stream = invoke_agent_response["completion"]

In [16]:
for event in event_stream:
    print(event)

{'trace': {'agentAliasId': 'N2YQNYIZYJ', 'agentId': '8SOPTO6T5R', 'agentVersion': '1', 'callerChain': [{'agentAliasArn': 'arn:aws:bedrock:us-west-2:092413168457:agent-alias/8SOPTO6T5R/N2YQNYIZYJ'}], 'sessionId': '481495f8-782d-4cf6-8fde-25092cf2449c', 'trace': {'orchestrationTrace': {'modelInvocationInput': {'foundationModel': 'anthropic.claude-3-haiku-20240307-v1:0', 'inferenceConfiguration': {'maximumLength': 2048, 'stopSequences': ['</invoke>', '</answer>', '</error>'], 'temperature': 0.0, 'topK': 250, 'topP': 1.0}, 'text': '{"system":" You are an advanced AI agent acting as a front line customer support agent. You have been provided with a set of functions to answer the user\'s question. You must call the functions in the format below: <function_calls>   <invoke>     <tool_name>$TOOL_NAME</tool_name>     <parameters>       <$PARAMETER_NAME>$PARAMETER_VALUE</$PARAMETER_NAME>       ...     </parameters>   </invoke> </function_calls> Here are the functions available: <functions>    </

In [17]:
message = "Hello, I bought a mug from your store yesterday, and it broke. I want to return it."

sessionId = str(uuid.uuid4())

In [18]:
invoke_agent_and_print(
    agentAliasId=agentAliasId,
    agentId=agentId,
    sessionId=sessionId,
    inputText=message,
    enableTrace=True,
)

User: Hello, I bought a mug from your store yesterday, and it broke. I want
to return it.

Agent: 
Agent's thought process:
  Okay, let's go through the steps to handle this return request:

Agent's thought process:
  Apologies, let me try that again in the correct format:

Agent's thought process:
  I apologize, it seems there was an issue with my previous function
  call. Let me try a different approach to get the return policy
  information.

Agent's thought process:
  I apologize, it seems there was an issue with my previous function
  call. Let me try a different approach to get the return policy
  information.

Observation:
  Type: FINISH

Final response:
  To return the broken mug, please follow these steps: 1. Gather the
  mug and your proof of purchase (such as the receipt). 2. Contact our
  customer service team at 1-800-123-4567 or
  customerservice@ourstore.com to initiate the return process. 3. Our
  team will provide you with a return shipping label and instructions
  on 